<a href="https://colab.research.google.com/github/stfnnnnnnn/karl-mangahas-flyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfnnnnnnn/1st-act/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass()

In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)


In [4]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [5]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:,} rows")

dim_clients            104 rows
dim_content            519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily             78,835,655 rows
fact_daily_sample      11,694,072 rows
fact_query_90d         2,414,248 rows


In [6]:
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_impressions ELSE 0 END) AS imp_last30,

        SUM(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
            THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,

        SUM(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_clicks ELSE 0 END) AS clk_last30,

        SUM(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
            THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,

        AVG(CASE
            WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
            THEN f.gsc_avg_position END) AS pos_prev30,

        -- Kept only for later inspection; not a Week 5 model feature.
        AVG(CASE
            WHEN f.report_date > b.end_d - INTERVAL 30 DAY
            THEN f.gsc_avg_position END) AS pos_last30

    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 60 DAY
      AND f.report_date <= b.end_d
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING imp_prev30 >= 100
)
SELECT * FROM windowed
""").df()

features["is_declining"] = (
    features["imp_last30"] < 0.8 * features["imp_prev30"]
).astype(int)

print(f"Rows: {len(features):,}")
print(f"Clients: {features['client_hash_id'].nunique():,}")
print(f"Decline base rate: {features['is_declining'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 82,025
Clients: 37
Decline base rate: 26.9%


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

>For this first learned-model comparison, I will test **Logistic Regression and Random Forest** rather than assume one method is already best.

>Random Forest is appropriate for this problem because the relationships between previous impressions, previous clicks, and previous search position may not be purely linear. For example, the importance of a weak historical position may differ depending on how much historical visibility or click activity a page already had. A Random Forest can represent these nonlinear relationships and interactions without requiring me to define them manually in advance.

>Random Forest also provides feature-importance estimates that can help me inspect which historical search signals the fitted model relies on most. These importance values will be treated as model diagnostics rather than evidence that a feature causes future decline. This is useful for a decision-support project because I need to understand the behavior of the ranking rather than rely only on a final performance score.

>To determine whether the additional model complexity is justified, I will compare the Random Forest with **Logistic Regression and the Week 4 transparent baseline** using the same client-grouped holdout and the same evaluation metrics. The classifiers' predicted probabilities will be used as ranking scores, since the practical goal is to decide which pages deserve earlier review rather than simply assign a binary class. I will therefore compare the methods using ROC-AUC, Average Precision, Precision@20, and Precision@50 alongside the decline base rate.

>At this stage, Random Forest is my primary modeling candidate rather than an assumed final winner. The executed comparison will determine whether it actually improves the review ranking. If Logistic Regression or the transparent baseline performs similarly or better, the simpler method would remain a reasonable choice.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

>I use a **grouped split by `client_hash_id`** so that pages from the same client do not appear in both training and testing.

>Pages from one client may share content strategy, search demand, or measurement patterns. A random row split could therefore make the test set too similar to the training set. Holding out whole clients gives me a more demanding first check of whether the ranking transfers to clients the model did not see during training.

>The **Week 4 baseline rule is reapplied to the same grouped test rows used for the learned models** so that the comparison is fair. This does not mean that the Week 4 notebook originally used this exact train/test partition; rather, I preserve its rule unchanged and evaluate that rule again on the Week 5 holdout. Logistic Regression and Random Forest are then evaluated on those same held-out pages. This is still a March development evaluation rather than a final future-time test.

In [7]:
feature_cols = [
    "imp_prev30",
    "clk_prev30",
    "pos_prev30",
]

X = features[feature_cols].copy()
y = features["is_declining"]
groups = features["client_hash_id"]

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

assert train_clients.isdisjoint(test_clients)

print("Training rows:", f"{len(train_idx):,}")
print("Test rows:", f"{len(test_idx):,}")
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Test base rate:", f"{y_test.mean():.1%}")


Training rows: 55,841
Test rows: 26,184
Training clients: 27
Test clients: 10
Client overlap: 0
Test base rate: 26.0%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

>The **Week 4 baseline rule is reproduced without changing its thresholds or scoring logic and then evaluated on the Week 5 grouped client holdout** alongside Logistic Regression and Random Forest. All three methods are therefore evaluated on the same held-out pages and with the same evaluation metrics. This keeps the comparison fair while preserving the Week 4 baseline as the transparent rule that was established before model training.

> Although Logistic Regression and Random Forest are trained as binary classifiers using the later decline label, their predicted probabilities are used to **rank pages by review priority** rather than simply assign positive or negative classes. This matches the practical objective of the project, where a content manager has limited review capacity and needs to know which pages should be inspected first. For this reason, I compare Precision@20 and Precision@50 alongside Average Precision and ROC-AUC, with the held-out decline base rate included as context.

> The comparison table below determines whether either learned model improves on the transparent Week 4 baseline. I do not assume in advance that Random Forest will perform best simply because it can represent nonlinear relationships. If Random Forest produces stronger Average Precision or top-of-queue precision, that would support using the more flexible model for further investigation. If Logistic Regression performs similarly, its simpler structure may be preferable. If neither learned model meaningfully improves on the Week 4 baseline, then the transparent rule remains an important result rather than something that should be discarded.

> I will interpret the methods based on the **executed results from this corrected feature set**. Previous results produced using `pos_last30` should not be carried forward because that field belongs to the later outcome period and is no longer part of the model. The comparison below therefore becomes the source of truth for the Week 5 model results.

In [8]:
features["is_visible"] = (
    features["imp_prev30"] >= 300
).astype(int)

features["is_weak_position"] = (
    features["pos_prev30"].notna()
    & (features["pos_prev30"] > 10)
).astype(int)

features["baseline_score"] = (
    features["is_visible"]
    * features["is_weak_position"]
    * features["imp_prev30"]
)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42,
    )),
])

rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    )),
])

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

lr_prob = lr.predict_proba(X_test)[:, 1]
rf_prob = rf.predict_proba(X_test)[:, 1]


In [10]:
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(scores))
    order = np.argsort(-scores)[:k]
    return float(y_true[order].mean())

base_rate = float(y_test.mean())
baseline_scores = features.iloc[test_idx]["baseline_score"].to_numpy()

results = pd.DataFrame([
    {
        "Model": "Week 4 Baseline",
        "ROC-AUC": roc_auc_score(y_test, baseline_scores),
        "Average Precision": average_precision_score(y_test, baseline_scores),
        "Precision@20": precision_at_k(y_test, baseline_scores, 20),
        "Precision@50": precision_at_k(y_test, baseline_scores, 50),
        "Base Rate": base_rate,
    },
    {
        "Model": "Logistic Regression",
        "ROC-AUC": roc_auc_score(y_test, lr_prob),
        "Average Precision": average_precision_score(y_test, lr_prob),
        "Precision@20": precision_at_k(y_test, lr_prob, 20),
        "Precision@50": precision_at_k(y_test, lr_prob, 50),
        "Base Rate": base_rate,
    },
    {
        "Model": "Random Forest",
        "ROC-AUC": roc_auc_score(y_test, rf_prob),
        "Average Precision": average_precision_score(y_test, rf_prob),
        "Precision@20": precision_at_k(y_test, rf_prob, 20),
        "Precision@50": precision_at_k(y_test, rf_prob, 50),
        "Base Rate": base_rate,
    },
])

display(results.style.format({
    "ROC-AUC": "{:.3f}",
    "Average Precision": "{:.3f}",
    "Precision@20": "{:.1%}",
    "Precision@50": "{:.1%}",
    "Base Rate": "{:.1%}",
}))

,Model,ROC-AUC,Average Precision,Precision@20,Precision@50,Base Rate
0,Week 4 Baseline,0.479,0.251,5.0%,6.0%,26.0%
1,Logistic Regression,0.532,0.263,40.0%,28.0%,26.0%
2,Random Forest,0.533,0.283,50.0%,48.0%,26.0%


> The Week 4 baseline, Logistic Regression, and Random Forest were evaluated using the **same grouped client holdout** and the **same evaluation metrics**. The Week 4 baseline uses its transparent historical rule, while the learned models use `imp_prev30`, `clk_prev30`, and `pos_prev30` as predictive features. Keeping the held-out pages and evaluation procedure consistent makes the comparison fair and allows the results to reflect differences in the ranking methods rather than differences in the test data.

> Although Logistic Regression and Random Forest were trained as binary classifiers using the later decline label, their predicted probabilities were used to **rank pages by review priority**. This matches the practical objective of deciding which pages should be inspected first. I therefore compare Precision@20 and Precision@50 alongside Average Precision and ROC-AUC, with the **26.0% decline base rate** included as context.

> The corrected comparison shows that **Random Forest produced the strongest ranking among the three methods**. It achieved a ROC-AUC of **0.533** and Average Precision of **0.283**, compared with **0.532 and 0.263** for Logistic Regression and **0.479 and 0.251** for the Week 4 baseline. The difference in ROC-AUC between the two learned models is very small, so the stronger evidence in favor of Random Forest comes from its performance near the top of the ranking rather than from overall discrimination alone.

> At the review capacities that matter most for this project, Random Forest achieved **50.0% Precision@20 and 48.0% Precision@50**. Logistic Regression achieved **40.0% and 28.0%**, while the Week 4 baseline achieved only **5.0% and 6.0%**. Relative to the 26.0% decline base rate, the Random Forest concentrated substantially more later-declining pages near the top of the review queue, while the transparent baseline performed poorly at these review depths.

> These results support keeping **Random Forest as the primary model for further validation**, but they do not establish that the model is already final or consistently reliable. Its ROC-AUC of **0.533** remains only slightly above 0.5, indicating weak overall discrimination despite the much stronger Precision@20 and Precision@50 results. This contrast means that the model appears more useful at the top of this particular held-out ranking than across the full set of pages, but I do not yet know whether that top-of-queue advantage is stable across different held-out clients or validation splits. The next step is therefore to inspect the model's errors and feature dependence and to test the robustness of its ranking before relying on it for recommendations.ns.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

> The Random Forest produced the strongest top-of-queue results in the model comparison, but it still makes important ranking errors. Its ROC-AUC of **0.533** also shows that overall separation between declining and non-declining pages remains weak. I therefore inspected both the pages that received high model scores but did not later meet the decline definition and the pages that did decline despite receiving relatively low scores. **Because this is still one grouped development split, these errors help explain the current ranking but do not yet show whether its Precision@20 and Precision@50 will remain equally strong under other client holdouts.**

> **High-scored non-declining pages** represent false review priorities: the model assigns them relatively high decline probabilities even though they do not later meet the defined decline outcome. These cases show that combinations of historical impressions, clicks, and search position can resemble the patterns the Random Forest associates with decline without necessarily being followed by an actual decline. From these features alone, I cannot determine whether an individual error was caused by competition, seasonality, search-intent changes, consolidation, or another factor outside the model.

> **Low-scored declining pages** show the opposite limitation. These pages later meet the decline definition even though the model assigns them relatively low probabilities. This suggests that some future declines are not strongly signaled by the three historical search features available to the Week 5 model. In these cases, the model may simply lack enough information to distinguish a page that will later decline from one that will remain relatively stable.

> The feature-importance results show which of the three historical inputs the fitted Random Forest relies on most: **previous search position (`pos_prev30`)**, **previous impressions (`imp_prev30`)**, and **previous clicks (`clk_prev30`)**, in the order shown by the executed importance table. These values describe the behavior of this fitted model; they do not show that any feature causes future decline.

> The feature-importance analysis can also be compared with the Week 4 baseline, which used historical visibility and historical search position as its transparent review signals. If the Random Forest places substantial importance on these same variables, it suggests that the learned model is using some of the same historical evidence while combining the signals in a more flexible way. This is consistent with its stronger Precision@20 and Precision@50 results, but the weak overall ROC-AUC means the model still requires further validation before I can treat the ranking as reliable.

In [11]:
rf_estimator = rf.named_steps["model"]

importance = (
    pd.Series(
        rf_estimator.feature_importances_,
        index=feature_cols,
    )
    .sort_values(ascending=False)
)

importance.to_frame(name="Importance")


,Importance
pos_prev30,0.536078
imp_prev30,0.409771
clk_prev30,0.054151


In [12]:
errors = features.iloc[test_idx][[
    "client_hash_id",
    "content_hash_id",
    "imp_prev30",
    "clk_prev30",
    "pos_prev30",
    "is_declining",
]].copy()

errors["Probability"] = rf_prob

high_scored_negatives = (
    errors[errors["is_declining"] == 0]
    .sort_values("Probability", ascending=False)
    .head(5)
)

low_scored_positives = (
    errors[errors["is_declining"] == 1]
    .sort_values("Probability", ascending=True)
    .head(5)
)

print("Highest-scored pages that did not later meet the decline definition:")
display(high_scored_negatives)

print("Lowest-scored pages that did later meet the decline definition:")
display(low_scored_positives)


Highest-scored pages that did not later meet the decline definition:


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,is_declining,Probability
34925,client_fef1a8f436438636,content_854e96530388d558,123.0,0.0,38.298768,0,0.936667
53621,client_62f4a7e64f5e0096,content_ed1b332b1cf85f2b,1256.0,0.0,0.294339,0,0.936667
72573,client_fef1a8f436438636,content_55184d9c3962a8f0,252.0,0.0,28.392392,0,0.936667
11662,client_62f4a7e64f5e0096,content_4c41556eec9710de,118.0,0.0,9.927103,0,0.923333
44747,client_65de48885f4ef01b,content_d3deb25effd3253f,100.0,0.0,5.603492,0,0.920000


Lowest-scored pages that did later meet the decline definition:


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,is_declining,Probability
54494,client_62f4a7e64f5e0096,content_0fcdf0deb75b03d2,100.0,0.0,7.522928,1,0.000000
13934,client_62f4a7e64f5e0096,content_499ff4ad3829b26f,232.0,1.0,5.884966,1,0.000000
54140,client_62f4a7e64f5e0096,content_1fe8a8a4e913881f,178.0,0.0,5.306667,1,0.000000
28547,client_62f4a7e64f5e0096,content_7f0267173a27b951,1840.0,0.0,26.709075,1,0.003333
70061,client_62f4a7e64f5e0096,content_209e4fd55498cc11,203.0,0.0,11.815664,1,0.003333


## Self-check

Before you submit, confirm each line honestly:

- [ - ] Every section above is filled — markdown thinking AND the code that backs it
- [ - ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ - ] No client names, URLs, or private queries anywhere
- [ - ] My claims use careful words: observed, measured, directional, decision-support
- [ - ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.